# 01 · Visium HD — Data loading

**Dataset:** GEO GSM9936742 / HD-02 · Human femoral head · Visium HD (barcodes indicate 56 µm aggregation).

**Goal:** Read the 10x HDF5 count matrix, align all barcodes with Parquet spatial positions, inspect the supplied tissue image, and save **unmodified raw counts** with full-resolution pixel coordinates to `results/adata/01_raw.h5ad`.

Run cells in order with **Run → Run All Cells**. The notebook works when launched from either the project root or `notebooks/`. No QC, normalization, clustering, or unverified image scaling is performed here.

## 0 · Imports and project paths

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import numpy as np
import pandas as pd
import scanpy as sc
from PIL import Image
from scipy import sparse

# Locate project root without depending on a particular launch directory.
ROOT = Path.cwd().resolve()
if not (ROOT / "data").is_dir() and (ROOT.parent / "data").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "data").is_dir():
    raise FileNotFoundError("Cannot find data/. Start JupyterLab from visium_hd_tutorial/ or notebooks/.")

PREFIX = "GSM9936742_HD-02"
H5 = ROOT / "data" / f"{PREFIX}_filtered_feature_bc_matrix.h5"
POSITIONS = ROOT / "data" / f"{PREFIX}_tissue_positions.parquet"
IMAGE = ROOT / "data" / f"{PREFIX}_tissue_hires_image.png"
OUTPUT = ROOT / "results" / "adata" / "01_raw.h5ad"

for path in (H5, POSITIONS, IMAGE):
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path}")

print("Python:", sys.executable)
print("Scanpy:", version("scanpy"))
print("Project root:", ROOT)
print("All three input files found.")

## 1 · Load the raw 10x count matrix

Keep the count matrix sparse and unchanged. Gene symbols can repeat; preserve original gene IDs in `adata.var["gene_ids"]`.

In [ ]:
# make_unique=True handles duplicate gene symbols while retaining gene_ids.
adata = sc.read_10x_h5(H5, gex_only=True)
adata.var_names_make_unique()

assert adata.obs_names.is_unique, "Count matrix barcodes are not unique"
assert adata.var_names.is_unique, "Gene names must be unique"
assert sparse.issparse(adata.X), "Expected a sparse count matrix"

print(adata)
print("Matrix shape (bins × genes):", adata.shape)
print("Nonzero count entries:", adata.X.nnz)
print("Sparse format:", adata.X.getformat())
print("First barcodes:", adata.obs_names[:3].tolist())

## 2 · Read and align spatial coordinates

import sys
import pandas as pd
import pyarrow as pa
from importlib.metadata import version

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)
print("Installed Pandas:", version("pandas"))
print("Installed PyArrow:", version("pyarrow"))
print("Pandas path:", pd.__file__)
print("PyArrow path:", pa.__file__)Match by barcode **rather than row order**. `obsm["spatial"]` stores **(x, y) = (pixel column, pixel row)** in **full-resolution image coordinates**.

In [ ]:
import sys
import pandas as pd
import pyarrow as pa
from importlib.metadata import version

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)
print("Installed Pandas:", version("pandas"))
print("Installed PyArrow:", version("pyarrow"))
print("Pandas path:", pd.__file__)
print("PyArrow path:", pa.__file__)

pos = pd.read_parquet(POSITIONS)
required_columns = [
    "barcode", "in_tissue", "array_row", "array_col",
    "pxl_row_in_fullres", "pxl_col_in_fullres",
]
missing_columns = sorted(set(required_columns) - set(pos.columns))
if missing_columns:
    raise ValueError(f"Missing columns in positions parquet: {missing_columns}")
if pos["barcode"].isna().any() or not pos["barcode"].is_unique:
    raise ValueError("Spatial position barcodes must be non-null and unique")

pos = pos.set_index("barcode")
missing_in_positions = adata.obs_names.difference(pos.index)
extra_in_positions = pos.index.difference(adata.obs_names)
print("Matrix barcodes:", adata.n_obs)
print("Position barcodes:", len(pos))
print("Missing in positions:", len(missing_in_positions))
print("Extra in positions:", len(extra_in_positions))
if len(missing_in_positions):
    raise ValueError(f"Missing coordinates for {len(missing_in_positions)} matrix barcodes")

aligned = pos.loc[adata.obs_names]
for column in ("in_tissue", "array_row", "array_col"):
    adata.obs[column] = aligned[column].to_numpy()

xy = aligned[["pxl_col_in_fullres", "pxl_row_in_fullres"]].to_numpy(dtype=np.float64)
if not np.isfinite(xy).all():
    raise ValueError("Spatial coordinates contain NaN or infinite values")
adata.obsm["spatial"] = xy

print("Spatial coordinates shape:", adata.obsm["spatial"].shape)
print("Tissue membership:", adata.obs["in_tissue"].value_counts().to_dict())
print("Fullres x range:", (xy[:, 0].min(), xy[:, 0].max()))
print("Fullres y range:", (xy[:, 1].min(), xy[:, 1].max()))
print(adata.obs.head())

## 3 · Inspect the supplied hires tissue image

The positions use **full-resolution pixels**, but the supplied PNG is a **hires image**. Since no `scalefactors_json.json` was provided, do **not** assume the two coordinate systems match. We record image dimensions only; no overlay or arbitrary rescaling is applied.

In [ ]:
with Image.open(IMAGE) as img:
    image_width, image_height = img.size
    image_mode = img.mode

print("Hires image dimensions (width, height):", (image_width, image_height))
print("Image mode:", image_mode)
print("Fullres spatial coordinate bounds (x, y):",
      (xy[:, 0].min(), xy[:, 0].max()), (xy[:, 1].min(), xy[:, 1].max()))
print("Image-to-coordinate scale: NOT verified")

## 4 · Validate and save the AnnData object

Save the raw sparse matrix, gene metadata, bin metadata, and aligned full-resolution coordinates. The image itself remains an external file, referenced by a relative path.

In [ ]:
assert adata.shape == (14400, 18085), f"Unexpected dataset shape: {adata.shape}"
assert adata.obsm["spatial"].shape == (adata.n_obs, 2)
assert adata.obs_names.equals(aligned.index), "Barcode order mismatch"
assert sparse.issparse(adata.X), "Raw counts should remain sparse"

adata.uns["dataset_id"] = PREFIX
adata.uns["tissue"] = "human femoral head"
adata.uns["spatial_coordinate_system"] = "fullres_pixels_xy"
adata.uns["hires_image_relative_path"] = str(IMAGE.relative_to(ROOT))
adata.uns["hires_image_width"] = int(image_width)
adata.uns["hires_image_height"] = int(image_height)
adata.uns["image_alignment_verified"] = False

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(OUTPUT)
print("Saved:", OUTPUT)
print("File exists:", OUTPUT.is_file())
print("File size (MiB):", round(OUTPUT.stat().st_size / (1024**2), 2))

## 5 · Reload test (pipeline handoff)

Verify that the next notebook can read the saved artifact without relying on in-memory variables from this notebook.

In [ ]:
check = sc.read_h5ad(OUTPUT)
assert check.shape == adata.shape
assert check.obs_names.equals(adata.obs_names)
assert check.var_names.equals(adata.var_names)
assert np.array_equal(check.obsm["spatial"], adata.obsm["spatial"])
assert sparse.issparse(check.X)

print("PASS — 01_data_loading completed")
print("AnnData:", check.shape)
print("Spatial coordinates:", check.obsm["spatial"].shape)
print("Spatial metadata:", check.obs.columns.tolist())
print("Output:", OUTPUT.relative_to(ROOT))

### Next: `02_quality_control.ipynb`

Input: `results/adata/01_raw.h5ad`. Compute QC metrics on raw counts and assess appropriate thresholds before filtering. **Important:** Image-to-coordinate registration is not yet verified; do not overlay bins onto the hires PNG until a trustworthy scale/transform is obtained.